# Data Completeness Inspection
Inspect the processed train/val/test CSVs for OhioT1DM, BrisT1D, and HUPA-UCM.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../dataset")

DATASETS = {
    "OhioT1DM": {"prefix": "ohio"},
    "BrisT1D":  {"prefix": "bris"},
    "HUPA-UCM": {"prefix": "hupa"},
}

SPLITS = ["train", "val", "test"]
FEATURE_COLS = ["carbs", "total_insulin", "steps", "glucose"]

## 1. Row counts and NaN summary per split

In [2]:
for name, cfg in DATASETS.items():
    print(f"\n{'='*70}")
    print(f"  {name}")
    print(f"{'='*70}")
    for split in SPLITS:
        path = BASE / name / f"{cfg['prefix']}_{split}.csv"
        df = pd.read_csv(path, parse_dates=["date"])
        print(f"\n  [{split}]  rows: {len(df):,}")
        print(f"  NaN counts per column:")
        nan_counts = df[FEATURE_COLS].isna().sum()
        for col, cnt in nan_counts.items():
            pct = 100 * cnt / len(df)
            print(f"    {col:20s}  {cnt:>8,}  ({pct:.2f}%)")


  OhioT1DM

  [train]  rows: 61,665
  NaN counts per column:
    carbs                        0  (0.00%)
    total_insulin                0  (0.00%)
    steps                        0  (0.00%)
    glucose                      0  (0.00%)

  [val]  rows: 13,214
  NaN counts per column:
    carbs                        0  (0.00%)
    total_insulin                0  (0.00%)
    steps                        0  (0.00%)
    glucose                      0  (0.00%)

  [test]  rows: 13,218
  NaN counts per column:
    carbs                        0  (0.00%)
    total_insulin                0  (0.00%)
    steps                        0  (0.00%)
    glucose                      0  (0.00%)

  BrisT1D

  [train]  rows: 409,046
  NaN counts per column:
    carbs                        0  (0.00%)
    total_insulin                0  (0.00%)
    steps                        0  (0.00%)
    glucose                      0  (0.00%)

  [val]  rows: 87,654
  NaN counts per column:
    carbs                  

## 2. Per-subject row counts and glucose completeness

In [3]:
for name, cfg in DATASETS.items():
    print(f"\n{'='*70}")
    print(f"  {name} — per-subject breakdown")
    print(f"{'='*70}")
    frames = []
    for split in SPLITS:
        path = BASE / name / f"{cfg['prefix']}_{split}.csv"
        df = pd.read_csv(path, parse_dates=["date"])
        df["split"] = split
        frames.append(df)
    all_df = pd.concat(frames, ignore_index=True)

    summary = (
        all_df.groupby(["subject", "split"])
        .agg(
            rows=("glucose", "size"),
            glucose_nan=("glucose", lambda x: x.isna().sum()),
            carbs_nan=("carbs", lambda x: x.isna().sum()),
            insulin_nan=("total_insulin", lambda x: x.isna().sum()),
            steps_nan=("steps", lambda x: x.isna().sum()),
        )
        .reset_index()
    )
    summary["glucose_pct"] = 100 * (1 - summary["glucose_nan"] / summary["rows"])
    print(summary.to_string(index=False))


  OhioT1DM — per-subject breakdown
 subject split  rows  glucose_nan  carbs_nan  insulin_nan  steps_nan  glucose_pct
     559  test  2085            0          0            0          0        100.0
     559 train  9725            0          0            0          0        100.0
     559   val  2084            0          0            0          0        100.0
     563  test  2243            0          0            0          0        100.0
     563 train 10462            0          0            0          0        100.0
     563   val  2242            0          0            0          0        100.0
     570  test  2207            0          0            0          0        100.0
     570 train 10298            0          0            0          0        100.0
     570   val  2207            0          0            0          0        100.0
     575  test  2174            0          0            0          0        100.0
     575 train 10143            0          0            0     

## 3. Timestamp regularity — gap detection
Check for gaps > 5 min (expected interval) within each subject.

In [4]:
for name, cfg in DATASETS.items():
    print(f"\n{'='*70}")
    print(f"  {name} — timestamp gaps")
    print(f"{'='*70}")
    for split in SPLITS:
        path = BASE / name / f"{cfg['prefix']}_{split}.csv"
        df = pd.read_csv(path, parse_dates=["date"])

        gap_rows = []
        for subj, grp in df.groupby("subject"):
            grp = grp.sort_values("date")
            diffs = grp["date"].diff()
            gaps = diffs[diffs > pd.Timedelta(minutes=5)]
            if len(gaps) > 0:
                gap_rows.append({
                    "subject": subj,
                    "n_gaps": len(gaps),
                    "max_gap": gaps.max(),
                    "median_gap": gaps.median(),
                    "total_gap_hrs": gaps.sum().total_seconds() / 3600,
                })

        if gap_rows:
            gap_df = pd.DataFrame(gap_rows)
            print(f"\n  [{split}]")
            print(gap_df.to_string(index=False))
        else:
            print(f"\n  [{split}]  No gaps > 5 min")


  OhioT1DM — timestamp gaps

  [train]
 subject  n_gaps         max_gap      median_gap  total_gap_hrs
     559      37 0 days 13:00:00 0 days 01:30:00      98.083333
     563      15 1 days 00:05:00 0 days 02:45:00      77.416667
     570      17 0 days 12:10:00 0 days 02:30:00      45.416667
     575      61 0 days 11:25:00 0 days 00:40:00      99.750000
     588       9 0 days 12:40:00 0 days 03:25:00      36.416667
     591      24 3 days 08:35:00 0 days 00:52:30     154.666667

  [val]
 subject  n_gaps         max_gap      median_gap  total_gap_hrs
     559       6 0 days 05:55:00 0 days 01:10:00      13.500000
     563       5 0 days 03:30:00 0 days 00:25:00       5.083333
     570       6 0 days 03:30:00 0 days 01:32:30      10.000000
     575      11 0 days 03:10:00 0 days 00:20:00       9.333333
     588       2 0 days 03:35:00 0 days 03:22:30       6.750000
     591       4 0 days 06:15:00 0 days 02:05:00      10.583333

  [test]
 subject  n_gaps         max_gap      median_

## 4. Value range sanity check

In [5]:
for name, cfg in DATASETS.items():
    print(f"\n{'='*70}")
    print(f"  {name} — value ranges (all splits combined)")
    print(f"{'='*70}")
    frames = []
    for split in SPLITS:
        path = BASE / name / f"{cfg['prefix']}_{split}.csv"
        frames.append(pd.read_csv(path))
    all_df = pd.concat(frames, ignore_index=True)

    stats = all_df[FEATURE_COLS].describe().T[["min", "25%", "50%", "mean", "75%", "max"]]
    print(stats.to_string())


  OhioT1DM — value ranges (all splits combined)
                min         25%         50%        mean    75%         max
carbs           0.0    0.000000    0.000000    0.640113    0.0  450.000000
total_insulin   0.0    0.058333    0.073333    0.156938    0.1   24.766667
steps           0.0    0.000000    0.000000    3.002826    0.0  160.000000
glucose        40.0  116.000000  156.000000  162.524694  202.0  400.000000

  BrisT1D — value ranges (all splits combined)
                   min    25%       50%        mean       75%        max
carbs           0.0000    0.0    0.0000    0.541896    0.0000   240.0000
total_insulin  -0.3078    0.0    0.0667    0.164606    0.1052    46.6663
steps           0.0000    0.0    0.0000   26.383535    3.0000  1830.0000
glucose        40.0000  115.0  142.0000  153.009944  180.0000   402.0000

  HUPA-UCM — value ranges (all splits combined)
                  min        25%         50%        mean         75%      max
carbs           0.000   0.000000    

## 5. Date range per subject

In [6]:
for name, cfg in DATASETS.items():
    print(f"\n{'='*70}")
    print(f"  {name} — date range per subject (all splits)")
    print(f"{'='*70}")
    frames = []
    for split in SPLITS:
        path = BASE / name / f"{cfg['prefix']}_{split}.csv"
        frames.append(pd.read_csv(path, parse_dates=["date"]))
    all_df = pd.concat(frames, ignore_index=True)

    range_df = (
        all_df.groupby("subject")["date"]
        .agg(["min", "max", "count"])
        .reset_index()
    )
    range_df["duration_days"] = (range_df["max"] - range_df["min"]).dt.total_seconds() / 86400
    print(range_df.to_string(index=False))


  OhioT1DM — date range per subject (all splits)
 subject                 min                 max  count  duration_days
     559 2021-12-07 01:15:00 2022-01-27 23:40:00  13894      51.934028
     563 2021-09-14 00:00:00 2021-11-07 08:15:00  14947      54.343750
     570 2021-12-07 16:30:00 2022-01-26 23:55:00  14712      50.309028
     575 2021-11-17 12:05:00 2022-01-11 10:30:00  14490      54.934028
     588 2021-08-30 11:55:00 2021-10-24 23:55:00  15983      55.500000
     591 2021-11-30 17:05:00 2022-01-23 21:15:00  14071      54.173611

  BrisT1D — date range per subject (all splits)
subject                 min                 max  count  duration_days
    P02 2023-06-02 00:00:00 2023-12-27 09:20:00  45770     208.388889
    P03 2023-06-07 00:00:00 2023-09-06 10:05:00  25935      91.420139
    P04 2023-06-16 00:00:00 2024-01-10 20:55:00  47185     208.871528
    P07 2023-06-02 08:00:00 2023-12-27 10:35:00  53906     208.107639
    P10 2023-06-23 00:00:00 2023-12-21 23:55:00  39240